# Notebook 04b — Piano Sub-type Analysis
## Finger Dexterity (piano_finger) vs Wrist Movement (piano_ankle)

Notebook 04 combined all piano sessions. This notebook splits them by `gameType` because they measure **different motor functions**:

| gameType | gameName | What it measures |
|---|---|---|
| `piano_finger` | Piano - Finger Dexterity | Individual finger tapping speed & accuracy |
| `piano_ankle` | Piano - Wrist Movement | Wrist/ankle range of motion & timing |

Also analyses `mobileMovements` — the per-finger data available in mobile mode.

In [ ]:
DATA_DIR    = '.'
OUT_DIR     = 'outputs'
GROUP_LABEL = 'Group A'

import os, json, sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings; warnings.filterwarnings('ignore')
sys.path.insert(0, os.path.dirname(os.path.abspath('hci_utils.py')))
from hci_utils import load_piano_sessions, save_fig

sessions, keypresses = load_piano_sessions(DATA_DIR)

# Attach gameType from raw file (hci_utils doesn't expose it yet)
raw = json.load(open(os.path.join(DATA_DIR, 'pianosessions.json')))
gt_map = {s['_id']: (s.get('gameType','unknown'), s.get('gameName','unknown')) for s in raw}
sessions['gameType'] = sessions['session_id'].map(lambda x: gt_map.get(x, ('unknown','unknown'))[0])
sessions['gameName'] = sessions['session_id'].map(lambda x: gt_map.get(x, ('unknown','unknown'))[1])

print('Piano sub-types found:')
print(sessions.groupby(['gameType','gameName']).agg(
    sessions=('session_id','count'),
    avg_score=('sessionScore','mean'),
    avg_hit_rate=('hit_rate','mean')
).round(2))


## 1 · Fig 04b-a — Session scores split by sub-type

In [ ]:
subtypes = sessions['gameType'].unique()
colors   = ['#2c4fa0', '#ba7517', '#2a6e4f', '#c84b2f']

fig, axes = plt.subplots(1, len(subtypes), figsize=(5*len(subtypes), 4), sharey=False)
if len(subtypes) == 1:
    axes = [axes]

for ax, subtype, color in zip(axes, subtypes, colors):
    sub = sessions[sessions['gameType'] == subtype].sort_values('time').reset_index(drop=True)
    name = sub['gameName'].iloc[0] if len(sub) else subtype
    sub['snum'] = range(1, len(sub)+1)
    ax.bar(sub['snum'], sub['sessionScore'], color=color, alpha=0.85, width=0.6)
    ax.set_xlabel('Session'); ax.set_ylabel('Score')
    ax.set_title(name, fontweight='bold')
    ax.set_xticks(sub['snum'])
    ax.set_xticklabels([f'S{i}' for i in sub['snum']])

fig.suptitle(f'Piano sub-type session scores — {GROUP_LABEL}', fontweight='bold')
plt.tight_layout()
save_fig(fig, 'fig04b_a_piano_subtype_scores.png', OUT_DIR); plt.show()


## 2 · Fig 04b-b — Per-finger accuracy (mobile mobileMovements)

This uses the `mobileMovements` array — available once mobile sessions are recorded.

In [ ]:
all_moves = []
for s in raw:
    for m in s.get('mobileMovements', []):
        all_moves.append({
            'session_id'     : s['_id'],
            'gameType'       : s.get('gameType', 'unknown'),
            'finger'         : m.get('finger', 'unknown'),
            'expectedFinger' : m.get('expectedFinger', 'unknown'),
            'responsetime'   : m.get('responsetime', 0),
            'correct'        : m.get('correct', 0),
        })

if not all_moves:
    print('No mobileMovements data yet.')
    print('This figure appears once mobile sessions are recorded.')
else:
    moves_df = pd.DataFrame(all_moves)
    facc = moves_df.groupby('finger').agg(
        total=('correct','count'),
        correct=('correct','sum'),
        avg_rt=('responsetime','mean')
    ).reset_index()
    facc['accuracy'] = facc['correct'] / facc['total'] * 100
    facc = facc.sort_values('accuracy')

    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    colors_bar = ['#2a6e4f' if v >= 80 else '#ba7517' if v >= 60 else '#c84b2f'
                  for v in facc['accuracy']]
    axes[0].barh(facc['finger'], facc['accuracy'], color=colors_bar, height=0.55)
    axes[0].axvline(80, color='#2a6e4f', ls='--', lw=1, alpha=0.5)
    axes[0].set_xlabel('Accuracy (%)'); axes[0].set_xlim(0, 110)
    axes[0].set_title('Per-finger accuracy', fontweight='bold')

    axes[1].barh(facc['finger'], facc['avg_rt'], color='#2c4fa0', height=0.55, alpha=0.85)
    axes[1].set_xlabel('Avg response time (s)')
    axes[1].set_title('Per-finger response time', fontweight='bold')

    fig.suptitle(f'Per-finger performance (mobile mode) — {GROUP_LABEL}', fontweight='bold')
    plt.tight_layout()
    save_fig(fig, 'fig04b_b_per_finger_accuracy.png', OUT_DIR); plt.show()
    print(facc[['finger','total','accuracy','avg_rt']].to_string(index=False))


## 3 · Fig 04b-c — Finger confusion matrix

Shows which finger the user actually used vs which was expected. Diagonal = correct. Off-diagonal = wrong finger.

In [ ]:
if not all_moves:
    print('No mobileMovements data yet.')
else:
    moves_df2 = pd.DataFrame(all_moves)
    pivot = moves_df2.groupby(['expectedFinger','finger']).size().unstack(fill_value=0)

    fig, ax = plt.subplots(figsize=(8, 6))
    im = ax.imshow(pivot.values, cmap='Blues', aspect='auto')
    ax.set_xticks(range(len(pivot.columns)))
    ax.set_xticklabels(pivot.columns, rotation=45, ha='right')
    ax.set_yticks(range(len(pivot.index)))
    ax.set_yticklabels(pivot.index)
    plt.colorbar(im, ax=ax, label='Count')
    ax.set_xlabel('Finger used'); ax.set_ylabel('Expected finger')
    ax.set_title(
        f'Finger confusion matrix — {GROUP_LABEL}\n'
        '(diagonal = correct, off-diagonal = wrong finger used)',
        fontweight='bold')
    plt.tight_layout()
    save_fig(fig, 'fig04b_c_finger_confusion_matrix.png', OUT_DIR); plt.show()


## 4 · Fig 04b-d — Wrist sub-type: response time over attempts

In [ ]:
ankle_sess = sessions[sessions['gameType'] == 'piano_ankle']
if ankle_sess.empty:
    print('No piano_ankle (Wrist Movement) sessions yet.')
else:
    ankle_kp = keypresses[keypresses['session_id'].isin(ankle_sess['session_id'])].copy()
    valid_kp  = ankle_kp[ankle_kp['responsetime'] > 0].reset_index(drop=True)
    valid_kp['idx'] = range(len(valid_kp))

    fig, ax = plt.subplots(figsize=(9, 4))
    outcome_map = {1: ('#2a6e4f', 'Hit'), 0: ('#c84b2f', 'Miss'), -1: ('#888780', 'Timeout')}
    for outcome, (color, label) in outcome_map.items():
        sub = valid_kp[valid_kp['correct'] == outcome]
        if not sub.empty:
            ax.scatter(sub['idx'], sub['responsetime'], color=color, alpha=0.6, s=40, label=label)
    ax.set_xlabel('Attempt index (chronological)')
    ax.set_ylabel('Response time (s)')
    ax.set_title(f'Wrist Movement — response time over attempts — {GROUP_LABEL}', fontweight='bold')
    ax.legend(); plt.tight_layout()
    save_fig(fig, 'fig04b_d_wrist_rt_over_time.png', OUT_DIR); plt.show()


## 5 · Fig 04b-e — Finger timeout config across sessions

Shows how timeout limits are set per finger — useful to know if difficulty is consistent across sessions.

In [ ]:
all_timeouts = []
for s in raw:
    ft = s.get('fingerTimeouts', {})
    for finger, limit in ft.items():
        all_timeouts.append({'session_id': s['_id'], 'finger': finger, 'limit_s': float(limit)})

if not all_timeouts:
    print('No fingerTimeouts data found.')
else:
    ft_df = pd.DataFrame(all_timeouts)
    ft_summary = ft_df.groupby('finger').agg(
        mean_limit=('limit_s','mean'),
        min_limit=('limit_s','min'),
        max_limit=('limit_s','max'),
        sessions=('session_id','nunique')
    ).reset_index().sort_values('mean_limit')

    fig, ax = plt.subplots(figsize=(10, 4))
    ax.barh(ft_summary['finger'], ft_summary['mean_limit'], color='#2c4fa0', height=0.55, alpha=0.85)
    ax.errorbar(
        ft_summary['mean_limit'], range(len(ft_summary)),
        xerr=[ft_summary['mean_limit']-ft_summary['min_limit'],
              ft_summary['max_limit']-ft_summary['mean_limit']],
        fmt='none', color='black', capsize=4, lw=1.5
    )
    ax.set_xlabel('Timeout limit (s)')
    ax.set_title(f'Finger timeout config across sessions — {GROUP_LABEL}\n'
                 '(error bars = min/max across sessions)', fontweight='bold')
    plt.tight_layout()
    save_fig(fig, 'fig04b_e_finger_timeout_config.png', OUT_DIR); plt.show()
    print(ft_summary.to_string(index=False))
